In [1]:
from pathlib import Path

In [3]:
from pathlib import Path

arquivo = Path("../Datasets Itens de casa/televisores.csv")

print("Caminho:", arquivo.resolve())
print("Arquivo existe:", arquivo.is_file())

Caminho: C:\Users\joaol\ProjetoTCS-2026\Datasets Itens de casa\televisores.csv
Arquivo existe: False


In [4]:
from pathlib import Path

pasta_atual = Path.cwd()

print("Pasta atual:", pasta_atual)
print("Existe a pasta de datasets?",
      (pasta_atual / "Datasets Itens de casa").is_dir())
print("Existe na pasta anterior?",
      (pasta_atual.parent / "Datasets Itens de casa").is_dir())
print("Existe na pasta atual?",
      (pasta_atual / "televisores.csv").is_file())

Pasta atual: c:\Users\joaol\ProjetoTCS-2026\Notebook_limpeza_dados
Existe a pasta de datasets? False
Existe na pasta anterior? False
Existe na pasta atual? False


In [5]:
from pathlib import Path

base = Path.cwd()

pastas = [
    base,
    base.parent,
    base.parent.parent,
    Path.home() / "ProjetoTCS-2026",
    Path.home() / "Datasets Itens de casa",
]

for pasta in pastas:
    print(f"\nPasta: {pasta}")
    print(f"Existe: {pasta.is_dir()}")

    if pasta.is_dir():
        print("Contém o CSV:", (pasta / "televisores.csv").is_file())
        print("Contém a pasta de datasets:",
              (pasta / "Datasets Itens de casa").is_dir())


Pasta: c:\Users\joaol\ProjetoTCS-2026\Notebook_limpeza_dados
Existe: True
Contém o CSV: False
Contém a pasta de datasets: False

Pasta: c:\Users\joaol\ProjetoTCS-2026
Existe: True
Contém o CSV: False
Contém a pasta de datasets: False

Pasta: c:\Users\joaol
Existe: True
Contém o CSV: False
Contém a pasta de datasets: False

Pasta: C:\Users\joaol\ProjetoTCS-2026
Existe: True
Contém o CSV: False
Contém a pasta de datasets: False

Pasta: C:\Users\joaol\Datasets Itens de casa
Existe: False


In [7]:

from pathlib import Path
import pandas as pd

# Caminho do arquivo original
arquivo = Path("../Datasets/televisores.csv")

if not arquivo.is_file():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo.resolve()}\n"
        "Confira o caminho da pasta Datasets Itens de casa."
    )

print("Carregando arquivo...", flush=True)

# Ler o CSV
df = pd.read_csv(arquivo, encoding="utf-8-sig")

print(f"Arquivo carregado: {len(df)} linhas.", flush=True)

# Padronizar nomes das colunas
df.columns = (
    df.columns.str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

# Remover colunas administrativas
colunas_remover = [
    "Código_de_Barras",
    "Status_do_Modelo",
    "Status_do_Registro",
    "Número_do_Registro"
]

df = df.drop(
    columns=[c for c in colunas_remover if c in df.columns]
)

# Remover linhas completamente duplicadas
df = df.drop_duplicates()

# Manter apenas um registro por marca e modelo,
# priorizando a linha com menos valores vazios
if "Modelo" in df.columns:
    colunas_identificacao = [
        c for c in ["Marca", "Modelo"] if c in df.columns
    ]

    if colunas_identificacao:
        df["_nulos"] = df.isnull().sum(axis=1)
        df = df.sort_values("_nulos")
        df = df.drop_duplicates(
            subset=colunas_identificacao,
            keep="first"
        )
        df = df.drop(columns="_nulos")

# Identificar a coluna de consumo
coluna_consumo = next(
    (c for c in df.columns if "Consumo" in c and "kWh" in c),
    None
)

if coluna_consumo is not None:
    df[coluna_consumo] = pd.to_numeric(
        df[coluna_consumo], errors="coerce"
    )

    # Remover consumos ausentes, zero ou negativos
    df = df.dropna(subset=[coluna_consumo])
    df = df[df[coluna_consumo] > 0]

    print(f"Coluna de consumo utilizada: {coluna_consumo}")

# Salvar o resultado na pasta do arquivo original
saida = arquivo.parent / "televisores_limpo.csv"

df.to_csv(saida, index=False, encoding="utf-8-sig")

print(f"Limpeza concluída: {len(df)} linhas restantes.")
print(f"Arquivo salvo em: {saida.resolve()}")


Carregando arquivo...
Arquivo carregado: 2094 linhas.
Coluna de consumo utilizada: Consumo_kWh_mês
Limpeza concluída: 1067 linhas restantes.
Arquivo salvo em: C:\Users\joaol\ProjetoTCS-2026\Datasets\televisores_limpo.csv


In [8]:

from pathlib import Path
import pandas as pd

# Caminho do arquivo original
arquivo = Path("../Datasets/televisores.csv")

if not arquivo.is_file():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo.resolve()}\n"
        "Confira o caminho da pasta Datasets Itens de casa."
    )

print("Carregando arquivo...", flush=True)

# Ler o CSV
df = pd.read_csv(arquivo, encoding="utf-8-sig")

print(f"Arquivo carregado: {len(df)} linhas.", flush=True)

# Padronizar nomes das colunas
df.columns = (
    df.columns.str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

# Remover colunas administrativas
colunas_remover = [
    "Código_de_Barras",
    "Status_do_Modelo",
    "Status_do_Registro",
    "Número_do_Registro"
]

df = df.drop(
    columns=[c for c in colunas_remover if c in df.columns]
)

# Remover linhas completamente duplicadas
df = df.drop_duplicates()

# Manter apenas um registro por marca e modelo,
# priorizando a linha com menos valores vazios
if "Modelo" in df.columns:
    colunas_identificacao = [
        c for c in ["Marca", "Modelo"] if c in df.columns
    ]

    if colunas_identificacao:
        df["_nulos"] = df.isnull().sum(axis=1)
        df = df.sort_values("_nulos")
        df = df.drop_duplicates(
            subset=colunas_identificacao,
            keep="first"
        )
        df = df.drop(columns="_nulos")

# Identificar a coluna de consumo
coluna_consumo = next(
    (c for c in df.columns if "Consumo" in c and "kWh" in c),
    None
)

if coluna_consumo is not None:
    df[coluna_consumo] = pd.to_numeric(
        df[coluna_consumo], errors="coerce"
    )

    # Remover consumos ausentes, zero ou negativos
    df = df.dropna(subset=[coluna_consumo])
    df = df[df[coluna_consumo] > 0]

    print(f"Coluna de consumo utilizada: {coluna_consumo}")

# Salvar o resultado na pasta do arquivo original
saida = arquivo.parent / "televisores_limpo.csv"

df.to_csv(saida, index=False, encoding="utf-8-sig")

print(f"Limpeza concluída: {len(df)} linhas restantes.")
print(f"Arquivo salvo em: {saida.resolve()}")


Carregando arquivo...
Arquivo carregado: 2094 linhas.
Coluna de consumo utilizada: Consumo_kWh_mês
Limpeza concluída: 1067 linhas restantes.
Arquivo salvo em: C:\Users\joaol\ProjetoTCS-2026\Datasets\televisores_limpo.csv
